# Adult Income Classifier

Binary classification model predicting whether income exceeds $50K/year using UCI Adult dataset.

In [1]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

## 1. Load data

In [2]:
df = pd.read_csv('../data/adult.csv')
df = df.dropna()

## 2. Quick EDA

In [3]:
print(df.shape)
print(df['income'].value_counts())

(30162, 15)
income
 <=50K    22654
 >50K      7508
Name: count, dtype: int64


## 3. Preprocessing

In [4]:
y = (df['income'] == ' >50K').astype(int)
X = df.drop('income', axis=1)
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

In [5]:
preprocess = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), numeric_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_cols)
    ]
)

## 4. Train/test split

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## 5. Train

In [7]:
pipeline = Pipeline([
    ('preprocess', preprocess),
    ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1))
])
pipeline.fit(X_train, y_train)

## 6. Evaluate

In [8]:
preds = pipeline.predict(X_test)
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

0.8532
              precision    recall  f1-score   support

           0       0.88      0.93      0.90      4531
           1       0.76      0.63      0.69      1502

    accuracy                           0.85      6033
   macro avg       0.82      0.78      0.80      6033
weighted avg       0.85      0.85      0.85      6033


## 7. Save model

In [9]:
os.makedirs('models', exist_ok=True)
joblib.dump(pipeline, 'models/adult_income_v1.joblib')
print('Model saved')

Model saved
